# Deploy Custom Python Backend Model to KServe with Triton Inference Server

Trains a **custom polynomial ridge regression model** entirely in pure NumPy (no standard ML
framework), packages it as a **Triton Python backend**, bakes it into a **Docker image** pushed
to the **Azure Container Registry (ACR)** associated with the AML workspace, and deploys it to
**KServe** using a custom container — no AzureML Online Endpoint, no blob storage initializer.

| Section | What happens |
|---------|-------------|
| **1 · Initial Setup & Configuration** | Install packages, set variables, connect to AML workspace |
| **2 · Model Training & Image Build** | Train model locally, write Triton Python backend, build & push Docker image to ACR |
| **3 · Inference Service Setup & Deployment** | Configure K8s client, deploy KServe `InferenceService` with custom container, wait for Ready |
| **4 · Inference Service Testing** | Send KFServing V2 inference requests and verify predictions |

**Key difference from the sklearn/pytorch notebooks:** the model is **baked into the Docker
image** rather than downloaded at pod startup by the KServe storage initializer. There is no
`storageUri` and no Azure Blob credential secret required.


---
## Section 1 — Initial Setup & Configuration

Install dependencies, set all configuration variables, import libraries, and connect to the
Azure ML workspace. **Run these cells once per kernel restart.**


### 1.1 · Install Required Packages

Installs KServe, Kubernetes, and Azure management SDKs.
`azure-mgmt-containerregistry` is installed `--user` to avoid conflicts with read-only system
packages; it provides the **ACR Tasks** API used in Section 2 to build the Docker image
inside Azure — no local Docker daemon required.


In [ ]:
%pip install "azure-ai-ml>=1.12.0" \
             "azure-identity>=1.14.0" \
             "numpy>=1.24.0" \
             "requests>=2.31.0" \
             "kubernetes>=28.1.0" \
             "kserve>=0.12.0" \
             "azure-storage-blob>=12.19.0"
# Install management SDKs to user site-packages (avoid system package conflicts)
%pip install "azure-mgmt-storage>=21.0.0" \
             "azure-mgmt-containerregistry>=10.0.0" \
             --user -q


In [ ]:
# ── Project Root Setup ────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

_nb_file = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb_file:
    _project_root = str(Path(_nb_file).resolve().parent)
    os.chdir(_project_root)
else:
    _project_root = os.getcwd()

if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

os.makedirs(os.path.join(_project_root, 'artifacts'), exist_ok=True)

print(f'Project root : {_project_root}')
print(f'Working dir  : {os.getcwd()}')


### 1.2 · Available Compute Targets (reference)

| Compute Name | Time Slicing Options | Base Node Guaranteed CPU | Base Node Guaranteed Memory |
|--------------|---------------------|--------------------------|------------------------------|
| cpu          | -2, -4              | 14                       | 46Gi                         |
| gput41       | -2, -4              | 6                        | 44Gi                         |
| gpuv1001     | -2, -4              | 4                        | 98Gi                         |
| gpuv1002     | -2, -4              | 10                       | 206Gi                        |
| gpua100      | -2, -4              | 22                       | 202Gi                        |
| gput44       | -2, -4, -8          | 56                       | 410Gi                        |
| gpuv1004     | -2, -4, -8          | 22                       | 422Gi                        |

> This notebook uses `KIND_CPU` — no GPU is required. Use `cpu-2` or `cpu-4`.


### 1.3 · Configuration

Set all deployment parameters here before running the notebook.

| Variable | Purpose |
|----------|---------|
| `triton_model_name` | Name of the Triton model (also used as the directory name under `/models`) |
| `inference_service_name` | Name of the KServe `InferenceService` resource to create |
| `kserve_namespace` | Kubernetes namespace where KServe CRDs are installed |
| `image_name` | Docker image name (without registry prefix or tag) |
| `image_version` | Docker image tag |
| `polynomial_degree` | Degree of the polynomial feature expansion used by the regression model |
| `request_cpu` / `request_ram` | CPU and memory resource requests for the predictor pod |
| `limit_cpu` / `limit_ram` | CPU and memory resource limits for the predictor pod |


In [ ]:
# ── KServe deployment settings ────────────────────────────────────────────────
triton_model_name      = 'poly_regressor'
inference_service_name = 'poly-regressor-triton'
kserve_namespace       = 'unified'
acr_pull_secret_name   = 'poly-regressor-triton-acr-pull'  # K8s docker-registry secret

# ── Docker image settings ──────────────────────────────────────────────────────
# Image will be pushed to: <acr_login_server>/<image_name>:<image_version>
image_name    = 'poly-regressor-triton'
image_version = 'v3'

# ── Model hyperparameters ─────────────────────────────────────────────────────
# Degree of polynomial feature expansion.
# Degree 13 gives RMSE < 0.03 over [-pi, pi] while all test cases pass (|err| < 0.05).
polynomial_degree = 13

# ── Resource requests / limits for the predictor pod ─────────────────────────
request_cpu = '500m'
request_ram = '2Gi'
limit_cpu   = '2'
limit_ram   = '4Gi'

tags = {
    'Purpose':   'Project Resources',
    'by_person': 'by_person',
}


### 1.4 · Imports & Workspace Connection

Imports all required libraries and connects to the Azure ML workspace using the pod's
**Managed Identity** (`AZURE_CLIENT_ID`). Workspace metadata is read from the Kubernetes
ConfigMap via `load_tags`.

Additional imports:
- `azure-mgmt-containerregistry` — ACR Tasks API for cloud-side Docker build
- `azure-mgmt-storage` — retrieve Azure storage account key for uploading the build context


In [ ]:
import os, re, json, tarfile, datetime, time, shutil
from pathlib import Path

# azure-mgmt-* packages installed --user; add user site-packages to sys.path
import sys
_user_site = os.path.expanduser('~/.local/lib/python3.11/site-packages')
if _user_site not in sys.path:
    sys.path.insert(0, _user_site)

import numpy as np

from azure.ai.ml import MLClient
from azure.identity import ManagedIdentityCredential
from azure.storage.blob import BlobServiceClient, generate_blob_sas, BlobSasPermissions

from kubernetes import client as k8s_client, config as k8s_config
from kserve import (
    KServeClient,
    V1beta1InferenceService,
    V1beta1InferenceServiceSpec,
    V1beta1PredictorSpec,
    constants,
)

# Service identity
nb_prefix              = os.getenv('NB_PREFIX', '/notebook/unknown/unknown')
_, namespace, pod_name = nb_prefix.strip('/').split('/')
AZURE_CLIENT_ID        = os.getenv('AZURE_CLIENT_ID')
credential             = ManagedIdentityCredential(client_id=AZURE_CLIENT_ID)


In [ ]:
from src._helpers.load_tags import load_tags
tags = load_tags(tags={}, namespace=namespace, pod_name=pod_name)
for k, v in tags.items():
    globals()[k] = v

print(tags)


In [ ]:
subscription_id = subscription_id
resource_group  = aml_workspace_rg
workspace       = aml_workspace

ml_client = MLClient(credential, subscription_id, resource_group, workspace)
print(f'ML Client workspace: {ml_client.workspace_name}')


### 1.5 · Resolve AML Workspace Storage & ACR

Retrieves:
- **Azure Blob Storage** account/container from the AML workspace's default datastore
  _(used to upload the Docker build context for ACR Tasks)_
- **Azure Container Registry** name and resource group from the workspace metadata
  _(the ACR Tasks build will push the image here)_
- **Storage account key** via the Azure Storage Management API
  _(needed to upload the build context tar.gz and to generate the SAS URL for ACR Tasks)_


In [ ]:
# ── Default datastore (for build context upload) ──────────────────────────────
default_ds      = ml_client.datastores.get_default()
storage_account = default_ds.account_name
container_name  = default_ds.container_name
print(f'Default datastore : {default_ds.name}')
print(f'Storage account   : {storage_account}')
print(f'Container         : {container_name}')

# ── ACR from workspace ────────────────────────────────────────────────────────
workspace_obj   = ml_client.workspaces.get(workspace)
acr_resource_id = workspace_obj.container_registry

# Parse /subscriptions/.../resourceGroups/<rg>/providers/.../registries/<name>
_acr_parts         = acr_resource_id.split('/')
acr_resource_group = _acr_parts[_acr_parts.index('resourceGroups') + 1]
acr_name           = _acr_parts[-1]
print(f'\nWorkspace ACR     : {acr_name}')
print(f'ACR resource group: {acr_resource_group}')

# ── Storage account key ───────────────────────────────────────────────────────
storage_account_key = None

# Method 1: from AML datastore credentials
try:
    ds_creds = default_ds.credentials
    if hasattr(ds_creds, 'account_key') and ds_creds.account_key:
        storage_account_key = ds_creds.account_key
        print('\nStorage key: retrieved from AML datastore credentials.')
except Exception:
    pass

# Method 2: Azure Storage Management API
if not storage_account_key:
    try:
        from azure.mgmt.storage import StorageManagementClient
        storage_mgmt = StorageManagementClient(credential, subscription_id)
        keys = storage_mgmt.storage_accounts.list_keys(resource_group, storage_account)
        storage_account_key = keys.keys[0].value
        print('\nStorage key: retrieved via Azure Storage Management API.')
    except Exception as e:
        print(f'\nWARNING: Could not retrieve storage key: {e}')

if storage_account_key:
    print(f'Storage key length: {len(storage_account_key)}')
else:
    raise RuntimeError(
        'Could not retrieve storage account key. '
        'Ensure the managed identity has Storage Account Contributor on the storage account.'
    )


---
## Section 2 — Model Training & Docker Image Build

Train the custom model locally, write the Triton Python backend files, build a Docker image
containing the model, and push it to the workspace ACR.

**Custom model:** degree-9 polynomial ridge regression  
**Target function:** `y = sin(x) + 0.5*sin(3x)` (sum of two harmonics, non-trivial shape)  
**Implementation:** pure NumPy closed-form solution — no scikit-learn, PyTorch, or TensorFlow

```
artifacts/
  poly_model_repo/
    poly_regressor/
      config.pbtxt          <- Triton Python backend config
      1/
        model.py            <- TritonPythonModel class (Python backend entry point)
        weights.npz         <- NumPy arrays: coef (n+1,) and degree (scalar)
  docker_build_context/
    Dockerfile              <- FROM tritonserver:23.08-py3, COPY model_repo /models
    model_repo/             <- copy of poly_model_repo/
```


### 2.1 · Train Polynomial Ridge Regression Model

Fits a degree-9 polynomial to the target `y = sin(x) + 0.5*sin(3x)` over `[-pi, pi]`
using the **closed-form ridge solution**:

```
w = (X^T X + alpha*I)^{-1} X^T y
```

where `X` is the Vandermonde matrix `[1, x, x^2, ..., x^9]` and `alpha = 1e-3`.

This is intentionally implemented in **plain NumPy** — no sklearn, no framework —
demonstrating that any arbitrary Python algorithm can be served via the Triton Python backend.


In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
degree  = polynomial_degree   # from configuration cell
alpha   = 1e-8                # ridge regularisation strength (small: near-exact fit)
x_scale = np.pi               # normalise x to [-1, 1] for numerical stability
n_train = 500

# ── Training data ─────────────────────────────────────────────────────────────
rng     = np.random.default_rng(42)
x_train = np.linspace(-np.pi, np.pi, n_train)
y_train = np.sin(x_train) + 0.5 * np.sin(3 * x_train) + rng.normal(0, 0.02, n_train)

# ── Polynomial features (normalised Vandermonde matrix) ───────────────────────
# Normalising x to [-1, 1] prevents x^13 from reaching ~10^6, which would make
# the normal equations ill-conditioned even with ridge regularisation.
def poly_features(x, deg, scale=1.0):
    x_norm = x.ravel() / scale
    return np.column_stack([x_norm ** i for i in range(deg + 1)])

X_train = poly_features(x_train, degree, x_scale)

# ── Closed-form ridge regression ──────────────────────────────────────────────
XTX  = X_train.T @ X_train
XTy  = X_train.T @ y_train
coef = np.linalg.solve(XTX + alpha * np.eye(degree + 1), XTy)

# ── Evaluate on a held-out grid ───────────────────────────────────────────────
x_eval = np.linspace(-np.pi, np.pi, 200)
y_true = np.sin(x_eval) + 0.5 * np.sin(3 * x_eval)
X_eval = poly_features(x_eval, degree, x_scale)
y_pred = X_eval @ coef
rmse   = np.sqrt(np.mean((y_pred - y_true) ** 2))

print(f'Model trained  — degree={degree}, ridge alpha={alpha}, x_scale={x_scale:.4f}')
print(f'Evaluation RMSE: {rmse:.6f}')
print(f'Coefficient shape: {coef.shape}')

# ── Quick sanity check ────────────────────────────────────────────────────────
x_chk = np.array([0.0, np.pi / 2, np.pi])
y_chk = np.sin(x_chk) + 0.5 * np.sin(3 * x_chk)
y_hat = poly_features(x_chk, degree, x_scale) @ coef
print('\nSanity check:')
for xv, yv, yh in zip(x_chk, y_chk, y_hat):
    err = abs(yh - yv)
    ok  = 'v' if err < 0.05 else 'x'
    print(f'  {ok}  x={xv:+.4f}  true={yv:+.4f}  pred={yh:+.4f}  |err|={err:.6f}')


### 2.2 · Prepare Triton Model Repository

Creates the Triton model repository layout on disk:

```
poly_model_repo/
  poly_regressor/
    config.pbtxt      <- backend: python, input x [FP32,-1,1], output y [FP32,-1,1]
    1/
      model.py        <- TritonPythonModel: load weights, run poly regression
      weights.npz     <- {coef: array(10,), degree: array(0-d)}
```

The `model.py` file uses `triton_python_backend_utils` — a module only available inside
the Triton container. It is **not imported** here; it is only written to disk.


In [ ]:
# ── Model repository paths ────────────────────────────────────────────────────
model_repo_dir = Path(_project_root) / 'artifacts' / 'poly_model_repo'
model_dir      = model_repo_dir / triton_model_name
version_dir    = model_dir / '1'
version_dir.mkdir(parents=True, exist_ok=True)
print(f'Model repository: {model_repo_dir}')

# ── Save model weights ────────────────────────────────────────────────────────
weights_path = version_dir / 'weights.npz'
np.savez(str(weights_path), coef=coef, degree=np.array(degree), x_scale=np.array(x_scale))
print(f'Saved weights    : {weights_path}  ({weights_path.stat().st_size} bytes)')

# ── Write model.py (Triton Python backend) ────────────────────────────────────
# Content uses single-quotes to avoid any triple-quote sequences.
model_py_content = "import triton_python_backend_utils as pb_utils\nimport numpy as np\nimport os\n\n\nclass TritonPythonModel:\n    # Degree-n polynomial ridge regression served via the Triton Python backend.\n    #\n    # Input:  x -- FP32 tensor, shape [-1, 1] (one scalar x per row)\n    # Output: y -- FP32 tensor, shape [-1, 1] (predicted y = sin(x) + 0.5*sin(3x))\n\n    def initialize(self, args):\n        # Load polynomial coefficients from weights.npz\n        model_dir    = os.path.join(args['model_repository'], args['model_version'])\n        weights_path = os.path.join(model_dir, 'weights.npz')\n        params       = np.load(weights_path)\n        self.coef    = params['coef'].astype(np.float64)  # shape (degree+1,)\n        self.degree  = int(params['degree'])\n        self.x_scale = float(params['x_scale'])  # normalisation factor (x divided by this before poly expansion)\n        print(f'[poly_regressor] Loaded weights: degree={self.degree}, x_scale={self.x_scale}')\n\n    def execute(self, requests):\n        # Run polynomial regression for each request in the batch\n        responses = []\n        for request in requests:\n            # Input: x -- FP32, shape [N, 1]\n            x = pb_utils.get_input_tensor_by_name(request, 'x').as_numpy().astype(np.float64)\n            # Normalise x the same way as during training, then build Vandermonde matrix\n            x_norm = x / self.x_scale  # normalise to [-1, 1]\n            X = np.column_stack([x_norm.ravel() ** i for i in range(self.degree + 1)])\n            # Apply linear model and reshape to [N, 1]\n            y = (X @ self.coef).reshape(-1, 1).astype(np.float32)\n            out_tensor = pb_utils.Tensor('y', y)\n            responses.append(pb_utils.InferenceResponse(output_tensors=[out_tensor]))\n        return responses\n\n    def finalize(self):\n        pass\n"
model_py_path = version_dir / 'model.py'
model_py_path.write_text(model_py_content)
print(f'Wrote model.py   : {model_py_path}')

# ── Write config.pbtxt (Triton Python backend config) ─────────────────────────
config_txt = 'name: "poly_regressor"\nbackend: "python"\nmax_batch_size: 0\n\n# max_batch_size: 0 means the batch dimension is explicit in dims.\n# Both input and output are [N, 1] where N is the number of inference requests.\n\ninput [\n  {\n    name:      "x"\n    data_type: TYPE_FP32\n    dims:      [-1, 1]\n  }\n]\n\noutput [\n  {\n    name:      "y"\n    data_type: TYPE_FP32\n    dims:      [-1, 1]\n  }\n]\n\ninstance_group [\n  {\n    kind:  KIND_CPU\n    count: 1\n  }\n]\n'
config_path = model_dir / 'config.pbtxt'
config_path.write_text(config_txt)
print(f'Wrote config.pbtxt: {config_path}')
print()
print('config.pbtxt:')
print(config_txt)
print('Model repository layout:')
for p in sorted(model_repo_dir.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(model_repo_dir)}  ({p.stat().st_size} bytes)')


### 2.3 · Build Docker Image Context

Creates a `Dockerfile` that:
1. Starts from the **Triton Inference Server 23.08** base image
2. Copies the model repository into `/models` inside the image
3. Exposes Triton ports (HTTP 8000, gRPC 8001, metrics 8002)

The build context is packaged as a **tar.gz** — the format expected by ACR Tasks when
`source_location` is a remote blob SAS URL.


In [ ]:
import shutil

# ── Build context directory ───────────────────────────────────────────────────
build_ctx_dir = Path(_project_root) / 'artifacts' / 'docker_build_context'
if build_ctx_dir.exists():
    shutil.rmtree(build_ctx_dir)
build_ctx_dir.mkdir(parents=True, exist_ok=True)

# ── Dockerfile ────────────────────────────────────────────────────────────────
# The tritonserver 23.08-py3 image includes:
#   - Python 3.8 + triton_python_backend_utils module
#   - ONNX Runtime, Python backend, and other standard backends
#   - ENTRYPOINT: /opt/tritonserver/bin/tritonserver
# KIND_CPU in config.pbtxt means no GPU driver is required on the AKS node.
dockerfile_lines = [
    'FROM nvcr.io/nvidia/tritonserver:23.08-py3',
    '',
    '# Copy the Triton model repository containing our Python backend model',
    'COPY model_repo /models',
    '',
    '# Triton HTTP (8080), gRPC (9000), and metrics (8002) ports',
    '# Port 8080 matches Knative Serving default user-port, avoiding H2C probe issues.',
    'EXPOSE 8080 9000 8002',
    '',
    '# Override the NVIDIA wrapper entrypoint and run tritonserver directly.',
    '# This avoids nvidia_entrypoint.sh exec argument handling issues on CPU nodes.',
    'ENTRYPOINT ["/opt/tritonserver/bin/tritonserver"]',
    f'CMD ["--model-repository=/models", "--model-control-mode=explicit", "--load-model={triton_model_name}", "--http-port=8080", "--grpc-port=9000"]',
]
dockerfile_content = '\n'.join(dockerfile_lines) + '\n'
(build_ctx_dir / 'Dockerfile').write_text(dockerfile_content)
print('Wrote Dockerfile:')
print(dockerfile_content)

# ── Copy model_repo into build context ───────────────────────────────────────
dst_model_repo = build_ctx_dir / 'model_repo'
if dst_model_repo.exists():
    shutil.rmtree(dst_model_repo)
shutil.copytree(str(model_repo_dir), str(dst_model_repo))
print('Copied model_repo into build context.')

# ── Create tar.gz build context ───────────────────────────────────────────────
tar_path = Path(_project_root) / 'artifacts' / 'docker_build_context.tar.gz'
with tarfile.open(str(tar_path), 'w:gz') as tar:
    tar.add(str(build_ctx_dir), arcname='.')

print('Build context layout:')
for p in sorted(build_ctx_dir.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(build_ctx_dir)}  ({p.stat().st_size} bytes)')
print(f'\nBuild context tar.gz: {tar_path}  ({tar_path.stat().st_size:,} bytes)')


### 2.4 · Upload Build Context to Azure Blob Storage

Uploads `docker_build_context.tar.gz` to the AML workspace's default datastore and
generates a **SAS URL** valid for 2 hours. ACR Tasks uses this SAS URL as the build context
source — it downloads the archive, extracts it, and runs `docker build` inside Azure.

This eliminates the need for a local Docker daemon or any Docker CLI tools.


In [ ]:
# ── Upload tar.gz to blob storage ─────────────────────────────────────────────
blob_service = BlobServiceClient(
    account_url=f'https://{storage_account}.blob.core.windows.net',
    credential=credential,
)
container_client = blob_service.get_container_client(container_name)

_ts       = datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S')
blob_name = f'build-contexts/{image_name}-{_ts}.tar.gz'

print(f'Uploading build context to blob://{storage_account}/{container_name}/{blob_name} ...')
with open(str(tar_path), 'rb') as f:
    container_client.upload_blob(blob_name, f, overwrite=True)
print('Upload complete.')

# ── Generate SAS URL (read, 2-hour expiry) ────────────────────────────────────
sas_expiry = datetime.datetime.utcnow() + datetime.timedelta(hours=2)
sas_token  = generate_blob_sas(
    account_name=storage_account,
    container_name=container_name,
    blob_name=blob_name,
    account_key=storage_account_key,
    permission=BlobSasPermissions(read=True),
    expiry=sas_expiry,
)
build_context_sas_url = (
    f'https://{storage_account}.blob.core.windows.net/{container_name}/{blob_name}?{sas_token}'
)
print(f'SAS URL generated (valid until {sas_expiry.isoformat()} UTC)')
print(f'SAS URL (truncated): {build_context_sas_url[:120]}...')


### 2.5 · Trigger ACR Tasks Build & Push

Uses the **Azure Container Registry Tasks REST API** to run a `DockerBuildRequest`
entirely inside Azure — no local Docker daemon required.

**How ACR Tasks work:**
1. ACR downloads the build context tar.gz from the SAS URL
2. Runs `docker build -t <acr>/<image>:<tag> .` inside an ephemeral Azure VM
3. Pushes the built image to the same ACR
4. Returns a `run_id` that can be polled for status and logs

The `scheduleRun` REST API is called directly with the Azure management token because
the `azure-mgmt-containerregistry>=14.0.0` SDK removed the `begin_schedule_run()` method
in favour of the newer `tasks` and `task_runs` operations. The run polling in Section 2.6
still uses the SDK's `runs.get()` which is available in all versions.

The final image will be at `<acr_login_server>/<image_name>:<image_version>`.


In [ ]:
import requests as _requests
from azure.mgmt.containerregistry import ContainerRegistryManagementClient

acr_mgmt = ContainerRegistryManagementClient(credential, subscription_id)

# ── Get ACR login server ───────────────────────────────────────────────────────
registry         = acr_mgmt.registries.get(acr_resource_group, acr_name)
acr_login_server = registry.login_server
full_image_name  = f'{acr_login_server}/{image_name}:{image_version}'
print(f'ACR login server : {acr_login_server}')
print(f'Target image     : {full_image_name}')

# ── Get Azure management bearer token ─────────────────────────────────────────
mgmt_token = credential.get_token('https://management.azure.com/.default').token

# ── Submit ACR scheduleRun via REST API ───────────────────────────────────────
# The azure-mgmt-containerregistry>=14 SDK removed begin_schedule_run().
# We call the REST API directly using the 2019-06-01-preview API version which
# still supports the DockerBuildRequest scheduleRun operation.
schedule_run_url = (
    f'https://management.azure.com/subscriptions/{subscription_id}'
    f'/resourceGroups/{acr_resource_group}'
    f'/providers/Microsoft.ContainerRegistry/registries/{acr_name}'
    f'/scheduleRun?api-version=2019-06-01-preview'
)

build_body = {
    'type': 'DockerBuildRequest',
    'imageNames': [f'{image_name}:{image_version}'],
    'isPushEnabled': True,
    'sourceLocation': build_context_sas_url,
    'dockerFilePath': 'Dockerfile',
    'platform': {'os': 'Linux', 'architecture': 'amd64'},
    'timeout': 1800,
}

print('\nSubmitting ACR Tasks build request ...')
run_resp = _requests.post(
    schedule_run_url,
    headers={'Authorization': f'Bearer {mgmt_token}', 'Content-Type': 'application/json'},
    json=build_body,
    timeout=60,
)

if run_resp.status_code not in (200, 202):
    raise RuntimeError(
        f'ACR scheduleRun failed: HTTP {run_resp.status_code}\n{run_resp.text[:400]}'
    )

run_data   = run_resp.json()
acr_run_id = run_data.get('name') or run_data.get('properties', {}).get('runId', '')
print(f'ACR build run submitted: run_id={acr_run_id}')
print(f'Initial status         : {run_data.get("properties", {}).get("status", "Queued")}')


### 2.6 · Wait for ACR Build to Complete

Polls the ACR run status every 15 seconds until the build succeeds or fails.
Typical build time for the `tritonserver:23.08-py3` base layer pull + copy: **4–8 minutes**
(the base layer is cached in ACR's build infrastructure after the first run).


In [ ]:
_ACR_BUILD_TIMEOUT = 900   # 15 min
_ACR_POLL_INTERVAL = 15

print(f'Polling ACR build run {acr_run_id} ...')
start = time.time()

while time.time() - start < _ACR_BUILD_TIMEOUT:
    run_status = acr_mgmt.runs.get(acr_resource_group, acr_name, acr_run_id)
    status     = run_status.status
    elapsed    = int(time.time() - start)
    print(f'[{elapsed:4d}s] {status}')
    if status in ('Succeeded', 'Failed', 'Canceled', 'Error', 'Timeout'):
        break
    time.sleep(_ACR_POLL_INTERVAL)

print(f'\nFinal status: {status}  (elapsed: {int(time.time() - start)}s)')

if status != 'Succeeded':
    try:
        log_result = acr_mgmt.runs.get_log_sas_url(acr_resource_group, acr_name, acr_run_id)
        print(f'\nBuild log URL:\n  {log_result.log_link}')
    except Exception:
        pass
    raise RuntimeError(
        f'ACR build failed with status {status!r}.\n'
        f'Check build logs via the URL above or:\n'
        f'  Azure Portal -> ACR -> {acr_name} -> Runs -> {acr_run_id}'
    )

print(f'\nDocker image successfully built and pushed:')
print(f'  {full_image_name}')


### 2.7 · Fetch Image Digest from ACR

KServe's underlying Knative Serving controller tries to resolve image tags to their
SHA-256 digest before scheduling pods. In this cluster, the Knative controller pods
cannot reach the workspace ACR directly (network policy), so tag resolution times out
and the ISVC never becomes Ready.

**Fix:** retrieve the digest directly from ACR using our notebook's managed identity,
then reference the image as `<registry>/<image>@sha256:<digest>`. With a digest-based
reference, Knative skips tag resolution entirely — it already has the immutable content
address — so the pod can be scheduled without the controller needing ACR access.

The sequence for managed-identity authentication against ACR:
1. Exchange AAD management token for an **ACR refresh token** via `/oauth2/exchange`
2. Exchange the refresh token for a repository-scoped **ACR access token** via `/oauth2/token`
3. Fetch the image manifest; the `Docker-Content-Digest` response header contains `sha256:...`


In [ ]:
# ── Step 1: exchange AAD token for ACR OAuth2 refresh token ─────────────────
aad_token = credential.get_token('https://management.azure.com/.default').token

exchange_resp = _requests.post(
    f'https://{acr_login_server}/oauth2/exchange',
    data={
        'grant_type':   'access_token',
        'service':      acr_login_server,
        'access_token': aad_token,
    },
    timeout=30,
)
exchange_resp.raise_for_status()
acr_refresh_token = exchange_resp.json()['refresh_token']
print(f'ACR refresh token obtained (length={len(acr_refresh_token)})')

# ── Step 2: exchange refresh token for repo-scoped access token ───────────────
scope_resp = _requests.post(
    f'https://{acr_login_server}/oauth2/token',
    data={
        'grant_type':    'refresh_token',
        'service':       acr_login_server,
        'scope':         f'repository:{image_name}:pull',
        'refresh_token': acr_refresh_token,
    },
    timeout=30,
)
scope_resp.raise_for_status()
acr_access_token = scope_resp.json()['access_token']
print(f'ACR scoped access token obtained')

# ── Step 3: fetch image manifest to get the content digest ───────────────────
manifest_resp = _requests.get(
    f'https://{acr_login_server}/v2/{image_name}/manifests/{image_version}',
    headers={
        'Authorization': f'Bearer {acr_access_token}',
        'Accept': 'application/vnd.docker.distribution.manifest.v2+json',
    },
    timeout=30,
)
manifest_resp.raise_for_status()
image_digest = manifest_resp.headers.get('Docker-Content-Digest', '')

if not image_digest:
    raise RuntimeError(
        f'Could not retrieve image digest from ACR. '
        f'Response headers: {dict(manifest_resp.headers)}'
    )

# Use digest-based reference: Knative skips tag resolution for @sha256: references
full_image_ref = f'{acr_login_server}/{image_name}@{image_digest}'
print(f'Image digest  : {image_digest}')
print(f'Image ref     : {full_image_ref}')
print()
print('Using digest-based image reference avoids Knative tag-to-digest resolution,')
print('which would otherwise time out (Knative controller cannot reach ACR in this cluster).')


---
## Section 3 — Inference Service Setup, Configuration & Deployment

Configure Kubernetes access and deploy the KServe `InferenceService` with the custom
ACR container image.

**Important difference from the sklearn/pytorch notebooks:** because the model is **baked
into the Docker image**, there is **no `storageUri`** and **no Azure Blob credential secret**
required. KServe starts the container directly — no storage initializer init-container runs.

```
InferenceService created
  +-- Predictor pod scheduled
        +-- container: kserve-container  (our custom ACR image)
              +-- /models/poly_regressor/config.pbtxt   (baked in)
              +-- /models/poly_regressor/1/model.py     (baked in)
              +-- /models/poly_regressor/1/weights.npz  (baked in)
              +-- /opt/tritonserver/bin/tritonserver
                    --model-repository=/models
                    --model-control-mode=explicit
                    --load-model=poly_regressor
                    --http-port=8080  ->  READY
```

**AKS -> ACR access:** a short-lived ACR access token is exchanged from the notebook pod's
Managed Identity and stored as a Kubernetes `docker-registry` secret (Section 3.2). The
ISVC `predictor.imagePullSecrets` references this secret so the kubelet can pull the image.

**Image reference:** the container uses a digest-based reference
(`acr.io/image@sha256:...`) fetched in Section 2.7. This bypasses Knative's tag-to-digest
resolution step, which times out because the Knative controller pods cannot reach the
private ACR endpoint in this cluster.


### 3.1 · Configure Kubernetes Client

Uses in-cluster config when running inside an AKS notebook pod. Falls back to
`az aks get-credentials` when running outside the cluster.


In [ ]:
try:
    k8s_config.load_incluster_config()
    print('Kubernetes client configured via in-cluster config.')
    _using_incluster = True
except k8s_config.ConfigException:
    _using_incluster = False
    print('Not running inside AKS -- falling back to az aks get-credentials...')
    import subprocess
    aks_compute_name = 'cpu-2'
    compute          = ml_client.compute.get(aks_compute_name)
    _parts           = compute.resource_id.split('/')
    _aks_rg          = _parts[_parts.index('resourceGroups') + 1]
    _aks_name        = _parts[-1]
    result = subprocess.run(
        ['az', 'aks', 'get-credentials', '--resource-group', _aks_rg,
         '--name', _aks_name, '--overwrite-existing'],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'az aks get-credentials failed: {result.stderr}')
    k8s_config.load_kube_config()
    print(result.stdout.strip() or 'kubeconfig updated.')

v1   = k8s_client.CoreV1Api()
pods = v1.list_namespaced_pod(namespace=kserve_namespace)
print(f'Connected to cluster. Pods in {kserve_namespace!r}: {len(pods.items)}')


### 3.2 · Create ACR Image Pull Secret

AKS node managed identities may not have `AcrPull` permissions pre-granted on the
workspace ACR. This cell creates a Kubernetes `docker-registry` secret from a
short-lived ACR access token obtained via Azure Managed Identity, so the kubelet can
pull our custom image.


In [ ]:
import base64 as _b64
import requests as _requests

# Get short-lived ACR access token via AAD -> ACR OAuth2 token exchange
_aad_token = credential.get_token('https://management.azure.com/.default').token
_exchange = _requests.post(
    f'https://{acr_login_server}/oauth2/exchange',
    data={'grant_type': 'access_token', 'service': acr_login_server, 'access_token': _aad_token},
)
_exchange.raise_for_status()
_refresh_token = _exchange.json()['refresh_token']

_scope_resp = _requests.post(
    f'https://{acr_login_server}/oauth2/token',
    data={
        'grant_type':    'refresh_token',
        'service':       acr_login_server,
        'scope':         f'repository:{image_name}:pull',
        'refresh_token': _refresh_token,
    },
)
_scope_resp.raise_for_status()
_acr_token = _scope_resp.json()['access_token']

# Build dockerconfigjson
_docker_cfg = {
    'auths': {
        acr_login_server: {
            'username': '00000000-0000-0000-0000-000000000000',
            'password': _acr_token,
            'auth':     _b64.b64encode(
                f'00000000-0000-0000-0000-000000000000:{_acr_token}'.encode()
            ).decode(),
        }
    }
}
_docker_cfg_b64 = _b64.b64encode(json.dumps(_docker_cfg).encode()).decode()

# Upsert the secret in Kubernetes
v1 = k8s_client.CoreV1Api()
try:
    v1.delete_namespaced_secret(acr_pull_secret_name, kserve_namespace)
    print(f'Deleted existing secret {acr_pull_secret_name!r}')
except k8s_client.exceptions.ApiException as _e:
    if _e.status != 404:
        raise
_secret = k8s_client.V1Secret(
    metadata=k8s_client.V1ObjectMeta(name=acr_pull_secret_name, namespace=kserve_namespace),
    type='kubernetes.io/dockerconfigjson',
    data={'.dockerconfigjson': _docker_cfg_b64},
)
v1.create_namespaced_secret(kserve_namespace, _secret)
print(f'Created ACR pull secret {acr_pull_secret_name!r} in namespace {kserve_namespace!r}')


### 3.3 · Deploy KServe InferenceService with Custom Container

Creates a KServe `InferenceService` using `predictor.containers` (the raw Kubernetes
container spec) instead of the built-in `predictor.triton` field. This is the
**KServe custom container** deployment pattern.

The tritonserver invocation is fully baked into the image. The Dockerfile sets:
```
ENTRYPOINT ["/opt/tritonserver/bin/tritonserver"]
CMD ["--model-repository=/models", "--model-control-mode=explicit",
     "--load-model=poly_regressor", "--http-port=8080", "--grpc-port=9000"]
```

Readiness and liveness probes hit Triton's built-in health endpoints:
- `GET /v2/health/ready` — 200 when all loaded models are ready
- `GET /v2/health/live` — 200 while the server process is alive


In [ ]:
ks_client = KServeClient()

isvc = V1beta1InferenceService(
    api_version=constants.KSERVE_V1BETA1,
    kind=constants.KSERVE_KIND_INFERENCESERVICE,
    metadata=k8s_client.V1ObjectMeta(
        name=inference_service_name,
        namespace=kserve_namespace,
        annotations={'sidecar.istio.io/inject': 'false'},
    ),
    spec=V1beta1InferenceServiceSpec(
        predictor=V1beta1PredictorSpec(
            # Use raw container spec -- the KServe 'custom container' pattern.
            # The model is already baked into the image; no storageUri is needed.
            # CMD in the Dockerfile already sets: tritonserver --model-repository=... etc.
            image_pull_secrets=[k8s_client.V1LocalObjectReference(name=acr_pull_secret_name)],
            containers=[
                k8s_client.V1Container(
                    name='kserve-container',
                    image=full_image_ref,   # digest ref: Knative skips tag resolution
                    # Do NOT declare ports here -- Knative adds containerPort 8080 (user-port)
                    # implicitly. Explicitly declaring ports triggers the queue-proxy's H2C
                    # upgrade probe, which Triton's HTTP server rejects with 405, blocking Ready.
                    resources=k8s_client.V1ResourceRequirements(
                        requests={'cpu': request_cpu, 'memory': request_ram},
                        limits={'cpu': limit_cpu,   'memory': limit_ram},
                    ),
                )
            ]
        )
    ),
)

# Delete existing InferenceService if present (clean redeploy)
try:
    ks_client.delete(name=inference_service_name, namespace=kserve_namespace)
    print(f'Deleted existing InferenceService {inference_service_name!r}. Waiting 10s ...')
    time.sleep(10)
except Exception:
    pass

ks_client.create(isvc)
print(f'InferenceService {inference_service_name!r} created in namespace {kserve_namespace!r}.')
print(f'Container image: {full_image_name}')
print(f'Triton will load model: {triton_model_name}')
print('Waiting for the service to become Ready ...')


### 3.3 · Wait for InferenceService Ready

Polls the `InferenceService` status every 15 seconds. The first deployment includes a
Docker image pull from ACR (~1–2 min) plus Triton Python backend startup.
Subsequent deployments skip the pull if the image is already cached on the node.


In [ ]:
_ISVC_TIMEOUT_SEC = 600   # 10 min
_POLL_INTERVAL    = 15

start = time.time()
ready = False

while time.time() - start < _ISVC_TIMEOUT_SEC:
    try:
        status_obj  = ks_client.get(name=inference_service_name, namespace=kserve_namespace)
        status_dict = status_obj if isinstance(status_obj, dict) else status_obj.to_dict()
        conditions  = status_dict.get('status', {}).get('conditions', []) or []

        ready_cond = next((c for c in conditions if c.get('type') == 'Ready'), None)
        if ready_cond and ready_cond.get('status') == 'True':
            ready = True
            break

        msg = ready_cond.get('message', 'waiting') if ready_cond else 'initializing'
        print(f'[{int(time.time() - start):4d}s] {msg}')

    except Exception as poll_err:
        print(f'[{int(time.time() - start):4d}s] Polling error: {poll_err}')

    time.sleep(_POLL_INTERVAL)

if not ready:
    raise TimeoutError(
        f'InferenceService {inference_service_name!r} did not become Ready within '
        f'{_ISVC_TIMEOUT_SEC // 60} minutes.\n'
        f'Diagnose with:\n'
        f'  kubectl describe inferenceservice {inference_service_name} -n {kserve_namespace}\n'
        f'  kubectl get pods -n {kserve_namespace}'
    )

print(f'\nInferenceService is Ready! (took {int(time.time() - start)}s)')


### 3.4 · Resolve Inference Endpoint URL

Discovers the **ClusterIP service** created by Knative for the predictor revision, yielding a
cluster-internal `svc.cluster.local` URL that works from inside the AKS pod.


In [ ]:
status_obj  = ks_client.get(name=inference_service_name, namespace=kserve_namespace)
status_dict = status_obj if isinstance(status_obj, dict) else status_obj.to_dict()
isvc_base_url = status_dict.get('status', {}).get('url', '')

if _using_incluster:
    all_svcs = k8s_client.CoreV1Api().list_namespaced_service(
        namespace=kserve_namespace,
        label_selector=f'serving.knative.dev/service={inference_service_name}-predictor',
    ).items
    clusterip_svcs = [
        s for s in all_svcs
        if s.spec.type == 'ClusterIP' and not s.metadata.name.endswith('-private')
    ]
    if clusterip_svcs:
        svc_name    = clusterip_svcs[0].metadata.name
        svc_host    = f'{svc_name}.{kserve_namespace}.svc.cluster.local'
        scoring_uri = f'http://{svc_host}/v2/models/{triton_model_name}/infer'
        print(f'In-cluster mode: using ClusterIP service {svc_name!r}')
    else:
        scoring_uri = (
            f'http://{inference_service_name}-predictor.{kserve_namespace}'
            f'.svc.cluster.local/v2/models/{triton_model_name}/infer'
        )
        print('In-cluster mode: using standard KServe cluster-internal URL (fallback)')
elif isvc_base_url:
    scoring_uri = f'{isvc_base_url}/v2/models/{triton_model_name}/infer'
    print(f'External URL: {isvc_base_url}')
else:
    scoring_uri = (
        f'http://{inference_service_name}.{kserve_namespace}'
        f'.svc.cluster.local/v2/models/{triton_model_name}/infer'
    )

print(f'\nTriton inference URL : {scoring_uri}')
print()
print('External URL (from outside the cluster):')
print(f'  {isvc_base_url}/v2/models/{triton_model_name}/infer' if isvc_base_url else '  (not available)')
print()
print('kubectl port-forward alternative:')
print(
    f'  kubectl port-forward -n {kserve_namespace} '
    f'svc/{inference_service_name}-predictor 8080:80'
)
print(f'  Then use: http://localhost:8080/v2/models/{triton_model_name}/infer')


---
## Section 4 — Inference Service Testing

Send live inference requests to the deployed custom Triton endpoint and verify the model's
predictions against the true target function `y = sin(x) + 0.5*sin(3x)`.


### 4.1 · Send Test Inference Requests

Sends requests using the **KFServing V2 inference protocol** (same JSON format used by all
Triton-backed InferenceServices in this project).

**Payload structure:**
```json
{"inputs": [{"name": "x", "shape": [N, 1], "datatype": "FP32", "data": [[x1], [x2], ...]}],
 "outputs": [{"name": "y"}]}
```

Test cases cover key points of `y = sin(x) + 0.5*sin(3x)`:

| x (radians) | True y | Notes |
|-------------|--------|-------|
| 0           | 0.000  | origin — both terms zero |
| pi/6 ~= 0.524 | 0.750  | sin(pi/6)=0.5, sin(pi/2)=1.0 -> 0.5+0.25=0.75 |
| pi/2 ~= 1.571 | 0.500  | sin(pi/2)=1, sin(3pi/2)=-1 -> 1.0-0.5=0.5 |
| pi ~= 3.142   | 0.000  | both terms vanish |
| -pi/4 ~= -0.785 | -0.854 | negative x |


In [ ]:
import requests as http_requests
import math

headers = {'Content-Type': 'application/json'}

# Reference function (same as training target)
def true_y(x):
    return math.sin(x) + 0.5 * math.sin(3 * x)

test_cases = [
    ('0',       0.0),
    ('pi/6',    math.pi / 6),
    ('pi/2',    math.pi / 2),
    ('pi',      math.pi),
    ('-pi/4',  -math.pi / 4),
]

print(f'{"x":12s}  {"true y":>10s}  {"pred y":>10s}  {"abs err":>10s}  pass')
print('-' * 56)

all_pass = True
for label, x_val in test_cases:
    payload = {
        'inputs': [{
            'name':     'x',
            'shape':    [1, 1],
            'datatype': 'FP32',
            'data':     [[x_val]],
        }],
        'outputs': [{'name': 'y'}],
    }
    resp = http_requests.post(
        scoring_uri, headers=headers, json=payload, verify=False, timeout=30
    )
    if resp.status_code != 200:
        print(f'{label:12s}  HTTP {resp.status_code}: {resp.text[:80]}')
        all_pass = False
        continue

    parsed = resp.json()
    if isinstance(parsed, str):
        parsed = json.loads(parsed)

    y_pred = parsed['outputs'][0]['data'][0]
    y_true = true_y(x_val)
    error  = abs(y_pred - y_true)
    ok     = error < 0.05
    all_pass = all_pass and ok
    print(f'{label:12s}  {y_true:>+10.4f}  {y_pred:>+10.4f}  {error:>10.6f}  {"v" if ok else "x"}')

print()
print('All test cases passed' if all_pass else 'Some test cases FAILED')


#### Batch inference

The Triton Python backend supports batching — send multiple `x` values in a single
request with shape `[N, 1]`.


In [ ]:
# ── Batch inference: all test x values in one request ─────────────────────────
x_values = [x for _, x in test_cases]
payload_batch = {
    'inputs': [{
        'name':     'x',
        'shape':    [len(x_values), 1],
        'datatype': 'FP32',
        'data':     [[xv] for xv in x_values],
    }],
    'outputs': [{'name': 'y'}],
}

resp_batch = http_requests.post(
    scoring_uri, headers=headers, json=payload_batch, verify=False, timeout=30
)
print(f'Batch HTTP status: {resp_batch.status_code}')

if resp_batch.status_code == 200:
    result = resp_batch.json()
    if isinstance(result, str):
        result = json.loads(result)

    y_preds = result['outputs'][0]['data']
    print(f'Batch response shape: {result["outputs"][0]["shape"]}')
    print()
    print('Batch predictions:')
    for (label, xv), yp in zip(test_cases, y_preds):
        print(f'  x={label:10s}  pred={yp:+.4f}  true={true_y(xv):+.4f}')

    print()
    print('Full KFServing V2 batch response:')
    print(json.dumps(result, indent=2))

# Equivalent curl command for a single inference
print()
print('Equivalent curl command (x=pi/2):')
print(
    f'curl -X POST {scoring_uri!r} '
    f'-H \'Content-Type: application/json\' '
    f'-d \'{{"inputs":[{{"name":"x","shape":[1,1],"datatype":"FP32",'
    f'"data":[[{math.pi/2:.4f}]]}}]}}\'' 
)


### 4.2 · Cleanup (Optional)

Uncomment to delete the KServe `InferenceService`.
Unlike the sklearn/pytorch notebooks, there are no patched secrets to restore here.

> Run this cell when you are done testing to free cluster resources.


In [ ]:
# ── Delete InferenceService ───────────────────────────────────────────────────
# ks_client.delete(name=inference_service_name, namespace=kserve_namespace)
# print(f'InferenceService {inference_service_name!r} deleted.')
